# Score Stability Experiments

这个 notebook 使用当前项目里的 `qwen3_ollama.py` + `src.scoring.pipeline` 评分链路，支持两个实验：

1. `data/experiments/same_pdf/` 里可以放多个 PDF；每个 PDF 默认跑 5 遍，统计每个文件自己的分数分布。
2. `data/experiments/group_a/` 和 `data/experiments/group_b/` 两组 PDF 做分布比较。

运行结果会自动写到 `data/experiments/results/`，解析后的 JSON 会缓存在 `data/experiments/results/parsed_cache/`。

默认复用解析缓存，只重复跑 scoring。需要连 parser 随机性一起测试时，把 `reparse_each_run=True`。


In [ ]:
!pip install matplotlib

In [1]:
from __future__ import annotations

import json
import sys
import time
from pathlib import Path

try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError:
    plt = None
    print('matplotlib is not installed; plotting helpers will be skipped.')
import pandas as pd

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from qwen3_ollama import _Scorer, score_application
from src.all_type_parser.all_type_parser import parse_and_save

CRITERIA_PATH = PROJECT_ROOT / 'criteria_points.json'
EXPERIMENT_ROOT = PROJECT_ROOT / 'data' / 'experiments'
SAME_PDF_DIR = EXPERIMENT_ROOT / 'same_pdf'
GROUP_A_DIR = EXPERIMENT_ROOT / 'group_a'
GROUP_B_DIR = EXPERIMENT_ROOT / 'group_b'
RESULTS_DIR = EXPERIMENT_ROOT / 'results'
PARSED_CACHE_DIR = RESULTS_DIR / 'parsed_cache'

DEFAULT_SAME_PDF_RUNS_PER_FILE = 3
EXPERIMENT_OLLAMA_MODEL = 'qwen3.5:27b'  # 改这里即可切换实验模型，例如 'qwen3.5:35b'

for path in [SAME_PDF_DIR, GROUP_A_DIR, GROUP_B_DIR, RESULTS_DIR, PARSED_CACHE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

SECTION_KEYS = [
    'general',
    'proposed_research',
    'training_development',
    'sites_support',
    'wpcc',
    'application_form',
]
SECTION_SCORE_COLUMNS = [f'{section_key}_score_100' for section_key in SECTION_KEYS]
RUN_SCORE_COLUMNS = ['overall_score_100', *SECTION_SCORE_COLUMNS]
RUN_DISPLAY_COLUMNS = ['pdf_name', 'run_idx', *RUN_SCORE_COLUMNS, 'avg_signal_score_0to5']
GROUP_RUN_DISPLAY_COLUMNS = ['group', *RUN_DISPLAY_COLUMNS]

print('Project root:', PROJECT_ROOT)
print('Same PDF dir:', SAME_PDF_DIR)
print('Group A dir:', GROUP_A_DIR)
print('Group B dir:', GROUP_B_DIR)
print('Results dir:', RESULTS_DIR)
print('Default same-pdf runs per file:', DEFAULT_SAME_PDF_RUNS_PER_FILE)
print('Experiment Ollama model:', EXPERIMENT_OLLAMA_MODEL)


Project root: d:\MSc_AI\SWE_group_project\nlp_grant_coursework
Same PDF dir: d:\MSc_AI\SWE_group_project\nlp_grant_coursework\data\experiments\same_pdf
Group A dir: d:\MSc_AI\SWE_group_project\nlp_grant_coursework\data\experiments\group_a
Group B dir: d:\MSc_AI\SWE_group_project\nlp_grant_coursework\data\experiments\group_b
Results dir: d:\MSc_AI\SWE_group_project\nlp_grant_coursework\data\experiments\results
Default same-pdf runs per file: 3
Experiment Ollama model: qwen3.5:27b


In [2]:
def list_pdfs(folder: Path) -> list[Path]:
    return sorted([path for path in folder.glob('*.pdf') if path.is_file()])


def parse_pdf_cached(pdf_path: Path, *, reparse: bool = False) -> tuple[dict, Path]:
    json_path = PARSED_CACHE_DIR / f'{pdf_path.stem}.json'
    if reparse or not json_path.exists():
        parse_and_save(str(pdf_path), str(json_path))
    parsed = json.loads(json_path.read_text(encoding='utf-8'))
    return parsed, json_path


def make_experiment_scorer(model_name: str | None = None) -> _Scorer:
    return _Scorer(model_name=model_name or EXPERIMENT_OLLAMA_MODEL)


def score_pdf_once(
    pdf_path: Path,
    *,
    scorer: _Scorer,
    run_tag: str,
    reparse: bool = False,
) -> dict:
    parsed, parsed_json_path = parse_pdf_cached(pdf_path, reparse=reparse)
    artifact_dir = RESULTS_DIR / run_tag
    artifact_dir.mkdir(parents=True, exist_ok=True)
    result = score_application(
        parsed,
        CRITERIA_PATH,
        doc_id=f'{pdf_path.stem}_{run_tag}',
        scorer=scorer,
        artifacts_dir=artifact_dir,
    )
    result['source_pdf'] = str(pdf_path)
    result['parsed_json'] = str(parsed_json_path)
    out_path = artifact_dir / f'{pdf_path.stem}_{run_tag}_scored.json'
    out_path.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
    result['result_json'] = str(out_path)
    return result


def average_signal_score(result: dict) -> float:
    scores = []
    for section in result.get('features', {}).values():
        for criterion in section.get('sub_criteria', section.get('criteria', [])):
            for signal in criterion.get('signals', []):
                scores.append(float(signal.get('score_0to5_raw', signal.get('score', 0)) or 0))
    return round(sum(scores) / len(scores), 4) if scores else 0.0


def flatten_result_row(result: dict, *, pdf_name: str, run_idx: int | None, group: str | None) -> dict:
    overall = result.get('overall', {})
    row = {
        'pdf_name': pdf_name,
        'run_idx': run_idx,
        'group': group,
        'overall_score_100': float(overall.get('final_score_0to100', 0)),
        'overall_score_10': float(overall.get('score_10', 0)),
        'quality_score_100': float(overall.get('quality_score_0to100', 0)),
        'coverage_score_100': float(overall.get('coverage_score_0to100', 0)),
        'avg_signal_score_0to5': average_signal_score(result),
        'total_items': int(overall.get('total_items', 0)),
        'signal_count': int(overall.get('signal_count', 0)),
        'good_items': int(overall.get('good_items', 0)),
        'positive_items': int(overall.get('positive_items', 0)),
        'pool_size': int(result.get('pool_size', 0)),
        'source_pdf': result.get('source_pdf'),
        'parsed_json': result.get('parsed_json'),
        'result_json': result.get('result_json'),
    }
    features = result.get('features', {})
    for section_key in SECTION_KEYS:
        section = features.get(section_key, {})
        section_overall = section.get('overall', {})
        row[f'{section_key}_score_100'] = float(section_overall.get('final_score_0to100', 0))
        row[f'{section_key}_score_10'] = float(section_overall.get('score_10', 0))
        row[f'{section_key}_quality_score_100'] = float(section_overall.get('quality_score_0to100', 0))
        row[f'{section_key}_coverage_score_100'] = float(section_overall.get('coverage_score_0to100', 0))
        row[f'{section_key}_signal_count'] = int(section_overall.get('signal_count', 0))
    return row


def score_columns(df: pd.DataFrame) -> list[str]:
    return [col for col in RUN_SCORE_COLUMNS if col in df.columns]


def score_mean_std_columns(summary: pd.DataFrame) -> list[str]:
    id_cols = [col for col in ['pdf_name', 'group'] if col in summary.columns]
    metric_cols = []
    for metric in RUN_SCORE_COLUMNS:
        for suffix in ['mean', 'std']:
            col = f'{metric}_{suffix}'
            if col in summary.columns:
                metric_cols.append(col)
    if metric_cols:
        return id_cols + metric_cols
    return [col for col in ['metric', 'mean', 'std'] if col in summary.columns]


def format_run_scores(row: dict) -> str:
    parts = [f"overall={float(row.get('overall_score_100', 0)):.1f}"]
    for section_key in SECTION_KEYS:
        col = f'{section_key}_score_100'
        parts.append(f"{section_key}={float(row.get(col, 0)):.1f}")
    return ' | '.join(parts)


def summarize_numeric(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    summary = pd.DataFrame({
        'mean': df[columns].mean(),
        'std': df[columns].std(ddof=1),
        'var': df[columns].var(ddof=1),
        'min': df[columns].min(),
        'max': df[columns].max(),
        'median': df[columns].median(),
    }).reset_index().rename(columns={'index': 'metric'})
    return summary


def summarize_distribution(df: pd.DataFrame, *, by: str = 'pdf_name') -> pd.DataFrame:
    columns = score_columns(df)
    summary = df.groupby(by)[columns].agg(['count', 'mean', 'std', 'var', 'min', 'max', 'median'])
    summary.columns = ['_'.join(col).strip('_') for col in summary.columns.to_flat_index()]
    return summary.reset_index()


def plot_same_pdf(df: pd.DataFrame, pdf_name: str) -> None:
    if plt is None:
        print('matplotlib is not installed; skipping plot.')
        return
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].plot(df['run_idx'], df['overall_score_100'], marker='o')
    axes[0].set_title(f'Overall score by run: {pdf_name}')
    axes[0].set_xlabel('Run index')
    axes[0].set_ylabel('Score / 100')
    axes[0].grid(alpha=0.3)

    axes[1].boxplot(df['overall_score_100'], vert=True)
    axes[1].set_title('Overall score distribution')
    axes[1].set_ylabel('Score / 100')
    plt.tight_layout()
    plt.show()


def plot_same_pdf_distribution(df: pd.DataFrame) -> None:
    if plt is None:
        print('matplotlib is not installed; skipping plot.')
        return
    pdf_names = sorted(df['pdf_name'].unique())
    fig, axes = plt.subplots(1, 2, figsize=(max(14, len(pdf_names) * 2.2), 4.5))
    for pdf_name, pdf_df in df.groupby('pdf_name'):
        axes[0].plot(pdf_df['run_idx'], pdf_df['overall_score_100'], marker='o', label=pdf_name)
    axes[0].set_title('Overall score by run')
    axes[0].set_xlabel('Run index')
    axes[0].set_ylabel('Score / 100')
    axes[0].grid(alpha=0.3)
    axes[0].legend(fontsize=8)

    grouped = [df.loc[df['pdf_name'] == pdf_name, 'overall_score_100'].tolist() for pdf_name in pdf_names]
    axes[1].boxplot(grouped, tick_labels=pdf_names)
    axes[1].set_title('Overall score distribution by PDF')
    axes[1].set_ylabel('Score / 100')
    axes[1].tick_params(axis='x', labelrotation=35)
    plt.tight_layout()
    plt.show()


def plot_group_compare(df: pd.DataFrame) -> None:
    if plt is None:
        print('matplotlib is not installed; skipping plot.')
        return
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    for group_name, group_df in df.groupby('group'):
        axes[0].hist(group_df['overall_score_100'], bins=min(10, max(3, len(group_df))), alpha=0.5, label=group_name)
    axes[0].set_title('Overall score distribution by group')
    axes[0].set_xlabel('Score / 100')
    axes[0].set_ylabel('Count')
    axes[0].legend()

    grouped = [group_df['overall_score_100'].tolist() for _, group_df in sorted(df.groupby('group'))]
    labels = [group_name for group_name, _ in sorted(df.groupby('group'))]
    axes[1].boxplot(grouped, tick_labels=labels)
    axes[1].set_title('Overall score boxplot')
    axes[1].set_ylabel('Score / 100')
    plt.tight_layout()
    plt.show()


def optional_stat_tests(group_a_scores: pd.Series, group_b_scores: pd.Series) -> pd.DataFrame:
    rows = []
    try:
        from scipy.stats import ks_2samp, ttest_ind
        t_stat, t_p = ttest_ind(group_a_scores, group_b_scores, equal_var=False)
        ks_stat, ks_p = ks_2samp(group_a_scores, group_b_scores)
        rows.append({'test': 'welch_ttest', 'statistic': float(t_stat), 'p_value': float(t_p)})
        rows.append({'test': 'ks_2samp', 'statistic': float(ks_stat), 'p_value': float(ks_p)})
    except Exception as exc:
        rows.append({'test': 'scipy_unavailable', 'statistic': None, 'p_value': None, 'note': str(exc)})
    return pd.DataFrame(rows)


def run_same_pdf_experiment(
    pdf_path: Path,
    *,
    n_runs: int = DEFAULT_SAME_PDF_RUNS_PER_FILE,
    reparse_each_run: bool = False,
    sleep_seconds: float = 0.0,
    model_name: str | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    scorer = make_experiment_scorer(model_name)
    rows = []
    for run_idx in range(1, n_runs + 1):
        run_tag = f'same_pdf_{pdf_path.stem}_run_{run_idx:02d}'
        print(f'[{pdf_path.name}] run {run_idx}/{n_runs}')
        result = score_pdf_once(
            pdf_path,
            scorer=scorer,
            run_tag=run_tag,
            reparse=reparse_each_run,
        )
        row = flatten_result_row(result, pdf_name=pdf_path.name, run_idx=run_idx, group='same_pdf')
        rows.append(row)
        print(f'  scores | {format_run_scores(row)}')
        if sleep_seconds > 0:
            time.sleep(sleep_seconds)
    df = pd.DataFrame(rows)
    summary = summarize_numeric(df, score_columns(df))
    timestamp = pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')
    df.to_csv(RESULTS_DIR / f'same_pdf_{pdf_path.stem}_runs_{timestamp}.csv', index=False)
    summary.to_csv(RESULTS_DIR / f'same_pdf_{pdf_path.stem}_summary_{timestamp}.csv', index=False)
    return df, summary


def run_same_pdf_distribution_experiment(
    pdf_paths: list[Path],
    *,
    runs_per_pdf: int = DEFAULT_SAME_PDF_RUNS_PER_FILE,
    reparse_each_run: bool = False,
    sleep_seconds: float = 0.0,
    model_name: str | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    assert pdf_paths, f'No PDF files found in {SAME_PDF_DIR}'
    scorer = make_experiment_scorer(model_name)
    rows = []
    total_runs = len(pdf_paths) * runs_per_pdf
    completed = 0
    for pdf_path in pdf_paths:
        for run_idx in range(1, runs_per_pdf + 1):
            completed += 1
            run_tag = f'same_pdf_{pdf_path.stem}_run_{run_idx:02d}'
            print(f'[{completed}/{total_runs}] {pdf_path.name} run {run_idx}/{runs_per_pdf}')
            result = score_pdf_once(
                pdf_path,
                scorer=scorer,
                run_tag=run_tag,
                reparse=reparse_each_run,
            )
            row = flatten_result_row(result, pdf_name=pdf_path.name, run_idx=run_idx, group='same_pdf')
            rows.append(row)
            print(f'  scores | {format_run_scores(row)}')
            if sleep_seconds > 0:
                time.sleep(sleep_seconds)
    df = pd.DataFrame(rows)
    summary = summarize_distribution(df, by='pdf_name')
    timestamp = pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')
    df.to_csv(RESULTS_DIR / f'same_pdf_distribution_runs_{timestamp}.csv', index=False)
    summary.to_csv(RESULTS_DIR / f'same_pdf_distribution_summary_{timestamp}.csv', index=False)
    return df, summary


def run_group_compare_experiment(
    *,
    group_a_paths: list[Path],
    group_b_paths: list[Path],
    runs_per_pdf: int = 1,
    reparse_each_run: bool = False,
    model_name: str | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    scorer = make_experiment_scorer(model_name)
    rows = []
    for group_name, pdf_paths in [('group_a', group_a_paths), ('group_b', group_b_paths)]:
        for pdf_path in pdf_paths:
            for run_idx in range(1, runs_per_pdf + 1):
                run_tag = f'{group_name}_{pdf_path.stem}_run_{run_idx:02d}'
                print(f'[{group_name}] {pdf_path.name} run {run_idx}/{runs_per_pdf}')
                result = score_pdf_once(
                    pdf_path,
                    scorer=scorer,
                    run_tag=run_tag,
                    reparse=reparse_each_run,
                )
                row = flatten_result_row(result, pdf_name=pdf_path.name, run_idx=run_idx, group=group_name)
                rows.append(row)
                print(f'  scores | {format_run_scores(row)}')
    df = pd.DataFrame(rows)
    score_cols = score_columns(df)
    summary = df.groupby('group')[score_cols].agg(['count', 'mean', 'std', 'var', 'min', 'max', 'median'])
    summary.columns = ['_'.join(col).strip('_') for col in summary.columns.to_flat_index()]
    summary = summary.reset_index()
    tests = optional_stat_tests(
        df.loc[df['group'] == 'group_a', 'overall_score_100'],
        df.loc[df['group'] == 'group_b', 'overall_score_100'],
    )
    timestamp = pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')
    df.to_csv(RESULTS_DIR / f'group_compare_runs_{timestamp}.csv', index=False)
    summary.to_csv(RESULTS_DIR / f'group_compare_summary_{timestamp}.csv')
    tests.to_csv(RESULTS_DIR / f'group_compare_tests_{timestamp}.csv', index=False)
    return df, summary, tests


## 实验 1：`same_pdf` 目录内多个 PDF，每个跑 5 遍

把一个或多个 PDF 放到 `data/experiments/same_pdf/`。下面这个 cell 会对目录里的每个 PDF 默认跑 5 遍，并输出每个 PDF 的分数分布。


In [ ]:
same_pdf_files = list_pdfs(SAME_PDF_DIR)
assert same_pdf_files, f'请在 {SAME_PDF_DIR} 里至少放 1 个 PDF。'

same_pdf_df, same_pdf_summary = run_same_pdf_distribution_experiment(
    same_pdf_files,
    runs_per_pdf=DEFAULT_SAME_PDF_RUNS_PER_FILE,
    reparse_each_run=False,
    sleep_seconds=0.0,
)

display(same_pdf_df[RUN_DISPLAY_COLUMNS])
display(same_pdf_summary[score_mean_std_columns(same_pdf_summary)])
plot_same_pdf_distribution(same_pdf_df)


[qwen3_ollama] using http://127.0.0.1:11434 model=qwen3.5:27b
[1/6] IC00001_DF_Doctoral.pdf run 1/3
[all_type_parser] ✓ fellowships_parser succeeded
[all_type_parser] saved → d:\MSc_AI\SWE_group_project\nlp_grant_coursework\data\experiments\results\parsed_cache\IC00001_DF_Doctoral.json
  scores | overall=91.2 | general=93.0 | proposed_research=97.5 | training_development=86.7 | sites_support=95.0 | wpcc=90.0 | application_form=85.0
[2/6] IC00001_DF_Doctoral.pdf run 2/3
  scores | overall=88.1 | general=92.0 | proposed_research=94.6 | training_development=80.0 | sites_support=95.0 | wpcc=82.2 | application_form=85.0
[3/6] IC00001_DF_Doctoral.pdf run 3/3


## 实验 2：两组 PDF 分布比较

把两组 PDF 分别放到：

- `data/experiments/group_a/`
- `data/experiments/group_b/`

建议每组 5 个。默认每个 PDF 跑 1 次。

In [ ]:
group_a_files = list_pdfs(GROUP_A_DIR)
group_b_files = list_pdfs(GROUP_B_DIR)

assert len(group_a_files) == 5, f'请在 {GROUP_A_DIR} 里放 5 个 PDF，当前有 {len(group_a_files)} 个。'
assert len(group_b_files) == 5, f'请在 {GROUP_B_DIR} 里放 5 个 PDF，当前有 {len(group_b_files)} 个。'

group_df, group_summary, group_tests = run_group_compare_experiment(
    group_a_paths=group_a_files,
    group_b_paths=group_b_files,
    runs_per_pdf=1,
    reparse_each_run=False,
)

display(group_df[GROUP_RUN_DISPLAY_COLUMNS])
display(group_summary[score_mean_std_columns(group_summary)])
display(group_tests)
plot_group_compare(group_df)


## 实验 3：Baseline —— 整个 parsed JSON 直接单次调用 LLM 打分（signal 级别）

**目的**：与实验 1（pipeline 多阶段打分）对比，评估两阶段流水线相对于最朴素 baseline 的增益。

**方法**：
- 不做任何预处理（无 chunking、无 Stage 1 证据检索、无分 section 调用）
- 把完整 parsed JSON + `criteria_points.json` 的所有 sub-criteria / signal 文本一次性喂给 LLM
- LLM 对**每个 signal** 输出 0–5 的整数分数
- 聚合逻辑与 pipeline 完全一致：signal → sub-criterion（0–10）→ section（0–100）→ overall（0–100）
- 与实验 1 在 overall、section、signal 三个层级做并排比较

In [5]:
# ── 实验 3 helpers ──────────────────────────────────────────────────────────
from src.scoring.pipeline import (
    load_rubric,
    OVERALL_EXCLUDED_SECTIONS_BY_DOC_TYPE,
    SECTION_EXCLUDED_SUB_IDS_BY_DOC_TYPE,
    SCORER_MAX_SCORE,
)

BASELINE_NUM_CTX = 32768   # Ollama context window for baseline (tokens). Increase if still failing.


class _BaselineScorer:
    """Thin wrapper around _Scorer that injects num_ctx into every request."""

    def __init__(self, model_name: str = EXPERIMENT_OLLAMA_MODEL, num_ctx: int = BASELINE_NUM_CTX):
        from qwen3_ollama import OLLAMA_HOST, OLLAMA_TIMEOUT
        import requests as _requests
        self.model_name = model_name
        self.num_ctx = num_ctx
        self._host = OLLAMA_HOST
        self._timeout = OLLAMA_TIMEOUT
        self._requests = _requests
        self.last_response_body = None
        print(f"[baseline] model={model_name}  num_ctx={num_ctx}", flush=True)

    def generate_json(self, messages: list, *, schema: dict, max_tokens: int) -> str:
        import re as _re
        payload = {
            "model": self.model_name,
            "messages": messages,
            "stream": False,
            "format": schema,
            "options": {
                "temperature": 0.6,
                "top_p": 0.9,
                "num_predict": max_tokens,
                "num_ctx": self.num_ctx,
            },
            "think": False,
        }
        try:
            resp = self._requests.post(
                f"{self._host}/api/chat", json=payload, timeout=self._timeout,
            )
        except self._requests.exceptions.ConnectionError as exc:
            raise RuntimeError(f"Could not connect to Ollama at {self._host}.") from exc
        except self._requests.exceptions.Timeout as exc:
            raise RuntimeError("Timed out waiting for Ollama.") from exc
        resp.raise_for_status()
        body = resp.json()
        self.last_response_body = body
        message = body.get("message") or {}
        content = _re.sub(r"<think>.*?</think>\s*", "", message.get("content") or "", flags=_re.DOTALL).strip()
        first, last = content.find("{"), content.rfind("}")
        return content[first:last + 1] if first != -1 and last > first else content

BASELINE_SECTION_KEYS = [
    "general", "proposed_research", "training_development",
    "sites_support", "wpcc", "application_form",
]


# ── Schema ───────────────────────────────────────────────────────────────────
def _build_baseline_signal_schema(rubric_sections: list) -> dict:
    """JSON schema：所有 sub-criterion → signal → integer(0-5)。"""
    properties = {}
    for section in rubric_sections:
        for sub in section['sub_criteria']:
            sig_props = {
                sig['sid']: {'type': 'integer', 'enum': [0, 1, 2, 3, 4, 5]}
                for sig in sub['signals']
            }
            properties[sub['sub_id']] = {
                'type': 'object',
                'properties': sig_props,
                'required': list(sig_props.keys()),
                'additionalProperties': False,
            }
    return {
        'type': 'object',
        'properties': properties,
        'required': list(properties.keys()),
        'additionalProperties': False,
    }


# ── Prompt ───────────────────────────────────────────────────────────────────
def _format_criteria_for_prompt(rubric_sections: list) -> str:
    """把所有 section/sub/signal 格式化成可读文字放进 prompt。"""
    lines = []
    for section in rubric_sections:
        lines.append(f"\n=== {section['human_name']} ===")
        for sub in section['sub_criteria']:
            lines.append(f"  [{sub['sub_id']}] {sub['name']}")
            lines.append(f"  Definition: {sub['definition']}")
            for sig in sub['signals']:
                lines.append(f"    ({sig['sid']}) {sig['text']}")
    return '\n'.join(lines)


def _build_baseline_messages(parsed_app: dict, rubric_sections: list) -> list:
    """单次调用 prompt：完整申请原文 + 全部 signal 文本 → 每个 signal 的 0-5 整数分。"""
    criteria_text = _format_criteria_for_prompt(rubric_sections)
    all_sub_ids = [
        sub['sub_id']
        for sec in rubric_sections
        for sub in sec['sub_criteria']
    ]
    system = (
        'You are an expert grant application reviewer.\n'
        'Score every signal in the rubric against the grant application.\n'
        'Each signal score MUST be an integer 0–5:\n'
        '  0=no evidence, 1=very weak, 2=weak, 3=moderate, '
        '4=strong with only minor gaps, 5=perfectly met.\n'
        'Return JSON only.\n'
        'Structure: top-level keys are sub-criterion IDs (e.g. "g.1", "pr.2").\n'
        'Each value is an object mapping signal IDs to integer scores '
        '(e.g. {"g.1.a": 4, "g.1.b": 3}).\n'
        f'Required top-level keys: {all_sub_ids}\n'
        'Do NOT omit any sub-criterion or signal.'
    )
    user = (
        'SCORING RUBRIC (read every signal carefully):\n'
        + criteria_text
        + '\n\nGRANT APPLICATION:\n'
        + json.dumps(parsed_app, ensure_ascii=False, indent=2)
        + '\n\nScore every signal listed above and return JSON only.'
    )
    return [
        {'role': 'system', 'content': system},
        {'role': 'user', 'content': user},
    ]


# ── Aggregation (mirrors pipeline logic exactly) ──────────────────────────────
def _aggregate_baseline(raw_scores: dict, rubric_sections: list, doc_type: str = '') -> tuple:
    """
    raw_scores: {sub_id: {sig_id: int(0-5), ...}, ...}
    Returns: (features_dict, overall_dict, avg_signal_score_0to5)
    Aggregation mirrors pipeline:
      signal (0-5, weighted) → sub score_10 → section score_10 → overall score_10
    """
    dt = doc_type.lower()
    excluded_sections = OVERALL_EXCLUDED_SECTIONS_BY_DOC_TYPE.get(dt, set())
    excluded_sub_ids  = SECTION_EXCLUDED_SUB_IDS_BY_DOC_TYPE.get(dt, set())

    features = {}
    all_signal_scores = []

    for section in rubric_sections:
        section_key = section['section_key']
        sub_results = []

        for sub in section['sub_criteria']:
            sub_id = sub['sub_id']
            sig_map = raw_scores.get(sub_id, {})
            signals = []
            for sig in sub['signals']:
                score = max(0, min(5, int(sig_map.get(sig['sid'], 0))))
                signals.append({'sid': sig['sid'], 'weight': sig['weight'], 'score': score})
                all_signal_scores.append(score)

            # sub score: same formula as pipeline _aggregate_sub_criterion
            total_w = sum(s['weight'] for s in signals) or 1.0
            weighted_sum = sum(s['score'] * s['weight'] for s in signals)
            score_10 = round((weighted_sum / (SCORER_MAX_SCORE * total_w)) * 10, 2)

            sub_results.append({
                'sub_id': sub_id,
                'name': sub['name'],
                'weight': sub['weight'],
                'score_10': score_10,
                'signals': signals,
                'counts_toward_section_average': sub_id not in excluded_sub_ids,
            })

        # section score: same formula as pipeline _aggregate_section
        scored = [s for s in sub_results if s['counts_toward_section_average']] or sub_results
        total_w = sum(s['weight'] for s in scored) or 1.0
        section_score_10 = round(
            sum(s['score_10'] * s['weight'] for s in scored) / total_w, 2
        )
        positive_items = sum(1 for s in scored if s['score_10'] > 0)
        signal_count = sum(len(s['signals']) for s in sub_results)

        features[section_key] = {
            'score_10': section_score_10,
            'weight': section['weight'],
            'sub_criteria': sub_results,
            'overall': {
                'score_10': section_score_10,
                'final_score_0to100': round(section_score_10 * 10, 2),
                'positive_items': positive_items,
                'scored_items': len(scored),
                'total_items': len(sub_results),
                'signal_count': signal_count,
            },
        }

    # overall score: same formula as pipeline _aggregate_overall
    scoring_features = {
        k: v for k, v in features.items() if k not in excluded_sections
    } or features
    section_weights = {k: v['weight'] for k, v in features.items()}
    total_w = sum(section_weights.get(k, 1.0) for k in scoring_features) or 1.0
    overall_10 = round(
        sum(
            features[k]['score_10'] * section_weights.get(k, 1.0)
            for k in scoring_features
        ) / total_w, 2
    )
    overall = {
        'score_10': overall_10,
        'final_score_0to100': round(overall_10 * 10, 2),
    }
    avg_signal = round(sum(all_signal_scores) / len(all_signal_scores), 4) if all_signal_scores else 0.0
    return features, overall, avg_signal


# ── Scoring entry point ───────────────────────────────────────────────────────
def baseline_score_pdf_once(
    pdf_path: Path,
    *,
    scorer,
    run_tag: str,
    reparse: bool = False,
    criteria_path: Path = CRITERIA_PATH,
) -> dict:
    """parse（带缓存）→ 单次 LLM 调用（signal 级分数）→ pipeline 相同的聚合逻辑。"""
    parsed, parsed_json_path = parse_pdf_cached(pdf_path, reparse=reparse)
    rubric_sections = load_rubric(criteria_path)
    doc_type = (parsed.get('doc_type') or '').lower()

    messages = _build_baseline_messages(parsed, rubric_sections)
    schema   = _build_baseline_signal_schema(rubric_sections)
    raw = scorer.generate_json(messages, schema=schema, max_tokens=4096)

    try:
        raw_scores = json.loads(raw)
    except json.JSONDecodeError:
        import re as _re
        m = _re.search(r'\{.*\}', raw, _re.DOTALL)
        raw_scores = json.loads(m.group()) if m else {}

    features, overall, avg_signal = _aggregate_baseline(raw_scores, rubric_sections, doc_type)

    artifact_dir = RESULTS_DIR / run_tag
    artifact_dir.mkdir(parents=True, exist_ok=True)
    out_path = artifact_dir / f'{pdf_path.stem}_{run_tag}_baseline.json'
    result = {
        'doc_id': f'{pdf_path.stem}_{run_tag}',
        'source_pdf': str(pdf_path),
        'parsed_json': str(parsed_json_path),
        'features': features,
        'overall': overall,
        'avg_signal_score_0to5': avg_signal,
        'raw_signal_scores': raw_scores,
        'raw_llm_response': raw,
    }
    out_path.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
    result['result_json'] = str(out_path)
    return result


def flatten_baseline_row(result: dict, *, pdf_name: str, run_idx: int) -> dict:
    overall = result.get('overall', {})
    features = result.get('features', {})
    row = {
        'pdf_name': pdf_name,
        'run_idx': run_idx,
        'overall_score_100': float(overall.get('final_score_0to100', 0)),
        'avg_signal_score_0to5': float(result.get('avg_signal_score_0to5', 0)),
        'source_pdf': result.get('source_pdf'),
        'result_json': result.get('result_json'),
    }
    for key in BASELINE_SECTION_KEYS:
        sec = features.get(key, {})
        row[f'{key}_score_100'] = float(
            sec.get('overall', {}).get('final_score_0to100', 0)
        )
    return row


def run_baseline_experiment(
    pdf_paths: list,
    *,
    runs_per_pdf: int = 3,
    reparse_each_run: bool = False,
    model_name: str | None = None,
) -> tuple:
    """对 same_pdf 目录里的 PDF 跑 baseline，每个 PDF 跑 runs_per_pdf 遍。"""
    assert pdf_paths, 'No PDF files found.'
    scorer = _BaselineScorer(model_name=model_name or EXPERIMENT_OLLAMA_MODEL)
    rows = []
    total = len(pdf_paths) * runs_per_pdf
    completed = 0
    for pdf_path in pdf_paths:
        for run_idx in range(1, runs_per_pdf + 1):
            completed += 1
            run_tag = f'baseline_{pdf_path.stem}_run_{run_idx:02d}'
            print(f'[baseline {completed}/{total}] {pdf_path.name} run {run_idx}/{runs_per_pdf}')
            result = baseline_score_pdf_once(
                pdf_path, scorer=scorer, run_tag=run_tag, reparse=reparse_each_run,
            )
            row = flatten_baseline_row(result, pdf_name=pdf_path.name, run_idx=run_idx)
            rows.append(row)
            section_parts = ' | '.join(
                f'{k}={row[f"{k}_score_100"]:.1f}' for k in BASELINE_SECTION_KEYS
            )
            print(f"  overall={row['overall_score_100']:.1f} | avg_sig={row['avg_signal_score_0to5']:.2f}")
            print(f'  sections: {section_parts}')

    df = pd.DataFrame(rows)
    score_cols = ['overall_score_100', 'avg_signal_score_0to5'] + [
        f'{k}_score_100' for k in BASELINE_SECTION_KEYS
    ]
    summary = summarize_numeric(df, score_cols)
    timestamp = pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')
    df.to_csv(RESULTS_DIR / f'baseline_runs_{timestamp}.csv', index=False)
    summary.to_csv(RESULTS_DIR / f'baseline_summary_{timestamp}.csv', index=False)
    return df, summary


def compare_pipeline_vs_baseline(pipeline_df: pd.DataFrame, baseline_df: pd.DataFrame) -> pd.DataFrame:
    """并排比较 pipeline（实验1）和 baseline（实验3）在同一组 PDF 上的均值分数。"""
    score_cols = ['overall_score_100', 'avg_signal_score_0to5'] + [
        f'{k}_score_100' for k in BASELINE_SECTION_KEYS
    ]
    p_rename = {c: f'pipeline_{c}' for c in score_cols}
    b_rename = {c: f'baseline_{c}' for c in score_cols}
    p_mean = pipeline_df.groupby('pdf_name')[
        [c for c in score_cols if c in pipeline_df.columns]
    ].mean().rename(columns=p_rename)
    b_mean = baseline_df.groupby('pdf_name')[
        [c for c in score_cols if c in baseline_df.columns]
    ].mean().rename(columns=b_rename)
    merged = p_mean.join(b_mean, how='outer')
    for c in score_cols:
        pc, bc = f'pipeline_{c}', f'baseline_{c}'
        if pc in merged.columns and bc in merged.columns:
            merged[f'diff_{c}'] = merged[pc] - merged[bc]
    return merged.reset_index()

def summarize_per_pdf(df: pd.DataFrame) -> pd.DataFrame:
    """对每个 PDF 分别统计多次 baseline 运行的 mean / std / min / max。"""
    score_cols = ['overall_score_100', 'avg_signal_score_0to5'] + [
        f'{k}_score_100' for k in BASELINE_SECTION_KEYS
    ]
    available = [c for c in score_cols if c in df.columns]
    agg = df.groupby('pdf_name')[available].agg(['mean', 'std', 'min', 'max'])
    agg.columns = ['_'.join(col) for col in agg.columns]
    return agg.reset_index()


In [6]:
# ── 实验 3 运行 ──────────────────────────────────────────────────────────────
# 与实验 1 使用相同模型和相同跑数，确保对比公平。
# 复用实验 1 的 parsed 缓存，仅重新调用 LLM。
# num_ctx 由 BASELINE_NUM_CTX 控制（默认 32768），可在 cell 9 顶部修改。
BASELINE_MODEL        = EXPERIMENT_OLLAMA_MODEL
BASELINE_RUNS_PER_PDF = DEFAULT_SAME_PDF_RUNS_PER_FILE

same_pdf_files = list_pdfs(SAME_PDF_DIR)
assert same_pdf_files, f'请在 {SAME_PDF_DIR} 里至少放 1 个 PDF。'

baseline_df, _ = run_baseline_experiment(
    same_pdf_files,
    runs_per_pdf=BASELINE_RUNS_PER_PDF,
    reparse_each_run=False,
    model_name=BASELINE_MODEL,
)

BASELINE_DISPLAY_COLS = [
    'pdf_name', 'run_idx', 'overall_score_100', 'avg_signal_score_0to5',
] + [f'{k}_score_100' for k in BASELINE_SECTION_KEYS]

print('\n── Baseline 各次运行分数 ──')
display(baseline_df[BASELINE_DISPLAY_COLS])

print('\n── Baseline 每个 PDF 的跨运行统计（mean / std / min / max）──')
baseline_per_pdf_summary = summarize_per_pdf(baseline_df)
display(baseline_per_pdf_summary)


[baseline] model=qwen3.5:27b  num_ctx=32768
[baseline 1/6] IC00001_DF_Doctoral.pdf run 1/3
  overall=97.1 | avg_sig=4.86
  sections: general=95.8 | proposed_research=100.0 | training_development=100.0 | sites_support=95.0 | wpcc=93.3 | application_form=98.3
[baseline 2/6] IC00001_DF_Doctoral.pdf run 2/3
  overall=98.4 | avg_sig=4.92
  sections: general=97.5 | proposed_research=100.0 | training_development=100.0 | sites_support=100.0 | wpcc=94.5 | application_form=98.3
[baseline 3/6] IC00001_DF_Doctoral.pdf run 3/3
  overall=98.0 | avg_sig=4.90
  sections: general=97.5 | proposed_research=100.0 | training_development=100.0 | sites_support=100.0 | wpcc=92.2 | application_form=98.3
[baseline 4/6] IC00147_after.pdf run 1/3
  overall=0.0 | avg_sig=0.00
  sections: general=0.0 | proposed_research=0.0 | training_development=0.0 | sites_support=0.0 | wpcc=0.0 | application_form=0.0
[baseline 5/6] IC00147_after.pdf run 2/3
  overall=0.0 | avg_sig=0.00
  sections: general=0.0 | proposed_research

,pdf_name,run_idx,overall_score_100,avg_signal_score_0to5,general_score_100,proposed_research_score_100,training_development_score_100,sites_support_score_100,wpcc_score_100,application_form_score_100
0,IC00001_DF_Doctoral.pdf,1,97.1,4.8571,95.8,100.0,100.0,95.0,93.3,98.3
1,IC00001_DF_Doctoral.pdf,2,98.4,4.9196,97.5,100.0,100.0,100.0,94.5,98.3
2,IC00001_DF_Doctoral.pdf,3,98.0,4.9018,97.5,100.0,100.0,100.0,92.2,98.3
3,IC00147_after.pdf,1,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,0.0
4,IC00147_after.pdf,2,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,0.0
5,IC00147_after.pdf,3,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,0.0



── Baseline 每个 PDF 的跨运行统计（mean / std / min / max）──


,pdf_name,overall_score_100_mean,overall_score_100_std,overall_score_100_min,overall_score_100_max,avg_signal_score_0to5_mean,avg_signal_score_0to5_std,avg_signal_score_0to5_min,avg_signal_score_0to5_max,general_score_100_mean,...,sites_support_score_100_min,sites_support_score_100_max,wpcc_score_100_mean,wpcc_score_100_std,wpcc_score_100_min,wpcc_score_100_max,application_form_score_100_mean,application_form_score_100_std,application_form_score_100_min,application_form_score_100_max
0,IC00001_DF_Doctoral.pdf,97.833333,0.665833,97.1,98.4,4.892833,0.0322,4.8571,4.9196,96.933333,...,95.0,100.0,93.333333,1.150362,92.2,94.5,98.3,0.0,98.3,98.3
1,IC00147_after.pdf,0.000000,0.000000,0.0,0.0,0.000000,0.0000,0.0000,0.0000,0.000000,...,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
# ── 实验 1 vs 实验 3 对比 ────────────────────────────────────────────────────
# 依赖：same_pdf_df（实验1）和 baseline_df（实验3）都已在内存中。

try:
    comparison_df = compare_pipeline_vs_baseline(same_pdf_df, baseline_df)
    print('── Pipeline（实验1）vs Baseline（实验3）均值对比 ──')
    display(comparison_df)

    if plt is not None:
        fig, axes = plt.subplots(1, 3, figsize=(20, 5))

        # ── 图1：overall 分数对比（每个 PDF 一组柱）──────────────────────────────
        pdf_names = comparison_df['pdf_name'].tolist()
        x = list(range(len(pdf_names)))
        w = 0.35
        axes[0].bar(
            [i - w/2 for i in x], comparison_df['pipeline_overall_score_100'],
            w, label='Pipeline（实验1）', color='steelblue',
        )
        axes[0].bar(
            [i + w/2 for i in x], comparison_df['baseline_overall_score_100'],
            w, label='Baseline（实验3）', color='darkorange',
        )
        axes[0].set_xticks(x)
        axes[0].set_xticklabels(pdf_names, rotation=30, ha='right', fontsize=8)
        axes[0].set_ylabel('Overall Score / 100')
        axes[0].set_title('Overall Score: Pipeline vs Baseline')
        axes[0].legend()
        axes[0].grid(axis='y', alpha=0.3)

        # ── 图2：各 section 分数差（pipeline - baseline）均值 ─────────────────────
        section_diffs = []
        for key in BASELINE_SECTION_KEYS:
            col = f'diff_{key}_score_100'
            if col in comparison_df.columns:
                section_diffs.append((key, comparison_df[col].mean()))
        if section_diffs:
            labels, vals = zip(*section_diffs)
            colors = ['steelblue' if v >= 0 else 'salmon' for v in vals]
            axes[1].bar(range(len(labels)), vals, color=colors)
            axes[1].axhline(0, color='black', linewidth=0.8)
            axes[1].set_xticks(range(len(labels)))
            axes[1].set_xticklabels(labels, rotation=35, ha='right', fontsize=8)
            axes[1].set_ylabel('Pipeline − Baseline (mean)')
            axes[1].set_title('Section-level Difference\n(+: pipeline higher)')
            axes[1].grid(axis='y', alpha=0.3)

        # ── 图3：avg_signal_score_0to5 对比 ──────────────────────────────────────
        p_sig_col = 'pipeline_avg_signal_score_0to5'
        b_sig_col = 'baseline_avg_signal_score_0to5'
        if p_sig_col in comparison_df.columns and b_sig_col in comparison_df.columns:
            axes[2].bar(
                [i - w/2 for i in x], comparison_df[p_sig_col],
                w, label='Pipeline（实验1）', color='steelblue',
            )
            axes[2].bar(
                [i + w/2 for i in x], comparison_df[b_sig_col],
                w, label='Baseline（实验3）', color='darkorange',
            )
            axes[2].set_xticks(x)
            axes[2].set_xticklabels(pdf_names, rotation=30, ha='right', fontsize=8)
            axes[2].set_ylabel('Avg Signal Score (0–5)')
            axes[2].set_title('Avg Signal Score: Pipeline vs Baseline')
            axes[2].legend()
            axes[2].grid(axis='y', alpha=0.3)
        else:
            axes[2].set_visible(False)

        plt.tight_layout()
        plt.show()

except NameError:
    print('same_pdf_df 未找到，请先运行实验 1 的 cell 以生成 same_pdf_df，再运行此对比 cell。')
